# LambdaG Visualization — Carmen in libros regum vs. Hildebert (H)

Token-level LambdaG visualization of the complete masked *Carmen in libros regum* (`X_cilr`) against Hildebert's test-corpus profile. The comparison uses up to 2000 candidate tokens and 20,000 reference tokens, matching the text quantities adopted in the global authorship-verification procedure.

The notebook generates the full local LambdaG HTML output and a second HTML file containing the 100 sentences with the most negative local contribution, shown with masked and unmasked text side by side.


In [2]:
from pathlib import Path
import numpy as np
import pandas as pd

from sentence_corpora import TwoLevelCorpus
from lambdag.visualize import lambdaG_visualize

# ---------------------------------------------------------
# Shared files produced by the general elegiac pipeline
# ---------------------------------------------------------
GENERAL_DIR = Path("..").resolve()

TEST_CORPUS_PATH = GENERAL_DIR / "test_corpus.pkl"
CILR_CORPUS_PATH = GENERAL_DIR / "cilr_corpus.pkl"
DATA_PROCESSED_DIR = GENERAL_DIR / "data_processed"

# Local HTML output directory
OUTPUT_DIR = Path("html_cilr")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for path in [
    TEST_CORPUS_PATH,
    CILR_CORPUS_PATH,
    DATA_PROCESSED_DIR,
]:
    if not path.exists():
        raise FileNotFoundError(f"Required path not found: {path}")

test_corpus = TwoLevelCorpus.from_pickle(str(TEST_CORPUS_PATH))
cilr_corpus = TwoLevelCorpus.from_pickle(str(CILR_CORPUS_PATH))

# Holdout sanity check
test_pairs = {(s.author, s.work) for s in test_corpus}
assert ("X", "cilr") not in test_pairs

print(f"General analysis directory: {GENERAL_DIR}")
print(f"HTML output directory: {OUTPUT_DIR.resolve()}")
print(f"test_corpus: {len(test_corpus)} sentences")
print(f"cilr_corpus: {len(cilr_corpus)} sentences")
print(f"Authors in test_corpus: {sorted(test_corpus.get_unique_values('author'))}")


General analysis directory: /Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/elegiac_couplets/Lexical analysis
HTML output directory: /Users/martinapaccara/Documents/Documenti/Università/POST-LAUREA/DOTTORATO/Lavoro/Digitale/Stilometria/elegiac_couplets/Lexical analysis/5. Carmen in libros regum/html_cilr
test_corpus: 15318 sentences
cilr_corpus: 791 sentences
Authors in test_corpus: ['AE', 'ASt', 'BB', 'BBi', 'BI', 'BM', 'BS', 'EM', 'H', 'HA', 'HM', 'HW', 'IS', 'M', 'NLC', 'PDE', 'PR', 'QS', 'RDV', 'RL', 'RT', 'SAC', 'SR', 'U1', 'U2', 'U3', 'U4']


In [3]:
# --- Candidate and reference sampling ---
rng = np.random.default_rng(seed=42)

# Hildebert candidate profile: up to 2000 tokens from the shared test corpus
known_corpus = test_corpus.filter_by_level("author", "H")
known_total_tokens = sum(len(s.text.split()) for s in known_corpus)
known_target_tokens = min(2000, known_total_tokens)

if known_target_tokens == 0:
    raise ValueError("No Hildebert material found in test_corpus")

print(f"Hildebert works in test_corpus: {known_corpus.get_unique_values('work')}")
for work in known_corpus.get_unique_values("work"):
    work_corpus = known_corpus.filter_by_level("work", work)
    print(
        f"  {work}: {len(work_corpus)} sentences, "
        f"{sum(len(s.text.split()) for s in work_corpus)} tokens"
    )

known_sentences_raw, known_breakdown = known_corpus.sample_balanced(
    tokens=known_target_tokens,
    rng=rng,
)
known_sentences = [list(s.text.split()) for s in known_sentences_raw]

print(
    f"\nKnown H sample: {sum(len(s) for s in known_sentences)} tokens "
    f"from {len(known_sentences)} sentences"
)
print(f"Known breakdown: {known_breakdown}")

# Reference population: 20,000 tokens from all other test-corpus authors
ref_corpus = TwoLevelCorpus(
    [s for s in test_corpus if s.author != "H"],
    levels=test_corpus.levels,
)
ref_total_tokens = sum(len(s.text.split()) for s in ref_corpus)
ref_target_tokens = min(20000, ref_total_tokens)

ref_sentences_raw, ref_breakdown = ref_corpus.sample_balanced(
    tokens=ref_target_tokens,
    rng=rng,
)
ref_sentences = [list(s.text.split()) for s in ref_sentences_raw]

print(
    f"Reference sample: {sum(len(s) for s in ref_sentences)} tokens "
    f"from {len(ref_sentences)} sentences"
)
print(f"Reference breakdown: {ref_breakdown}")


Hildebert works in test_corpus: ['carmina', 'mysterio']
  carmina: 83 sentences, 1230 tokens
  mysterio: 91 sentences, 1290 tokens

Known H sample: 2000 tokens from 140 sentences
Known breakdown: {'H': {'carmina': 1230, 'mysterio': 770}}
Reference sample: 20211 tokens from 1562 sentences
Reference breakdown: {'AE': {'bhc': 1480}, 'ASt': {'troilus': 7003}, 'BB': {'adelae': 255}, 'BBi': {'speculum': 378}, 'BI': {'reynardus': 363}, 'BM': {'regum': 194}, 'BS': {'mathematicus': 156}, 'EM': {'mahumeti': 206}, 'HA': {'pentateuchum': 177}, 'HM': {'gestis': 1007}, 'HW': {'hortus': 1565}, 'IS': {'entheticus': 282}, 'M': {'ars': 284, 'epistulae': 210, 'tobias': 158}, 'NLC': {'miracula': 602, 'speculum': 333}, 'PDE': {'rebussiculis': 226}, 'PR': {'aurora': 1575}, 'QS': {'alexandri': 426}, 'RDV': {'paulino': 123}, 'RL': {'anselmi': 782}, 'RT': {'epistulae': 474, 'memorabilibus': 500, 'miracula': 28}, 'SAC': {'excidio': 88}, 'SR': {'draco': 371}, 'U1': {'pyramo': 58}, 'U2': {'ysengrimus': 546}, 'U3'

In [4]:
# --- Prepare complete questioned text: Carmen in libros regum ---
q_sentences = [s.text.split() for s in cilr_corpus]
print(f"Questioned sentences (Carmen in libros regum / X_cilr): {len(q_sentences)}")

# Original unmasked sentences for colour-coding
quesplit_files = list(
    DATA_PROCESSED_DIR.glob("*cilr_sentences.quesplit.txt")
)

if not quesplit_files:
    raise FileNotFoundError(
        "No CILR .quesplit.txt file found in "
        f"{DATA_PROCESSED_DIR}"
    )

quesplit_file = quesplit_files[0]
with quesplit_file.open("r", encoding="utf-8") as f:
    unmasked_sents = [line.strip().split() for line in f if line.strip()]

print(
    f"Unmasked sentences loaded from {quesplit_file.name}: "
    f"{len(unmasked_sents)}"
)

if len(q_sentences) != len(unmasked_sents):
    raise ValueError(
        "Masked/unmasked sentence count mismatch: "
        f"{len(q_sentences)} vs {len(unmasked_sents)}"
    )


Questioned sentences (Carmen in libros regum / X_cilr): 791
Unmasked sentences loaded from X_cilr_sentences.quesplit.txt: 791


In [5]:
# --- Run local LambdaG visualization ---
FULL_HTML_PATH = OUTPUT_DIR / "cilr_vs_hildebert_full.html"

result = lambdaG_visualize(
    q_text=q_sentences,
    k_text=known_sentences,
    ref_text=ref_sentences,
    N=8,
    r=30,
    output="html",
    scale="relative",
    negative=True,
    order_by="importance",
    unmasked_sentences=unmasked_sents,
    rng=rng,
    print_path=str(FULL_HTML_PATH),
)

print(f"Full HTML saved to: {FULL_HTML_PATH}")
print(f"Table shape: {result['table'].shape}")
print(f"Sentences: {result['table']['sentence_id'].nunique()}")


Full HTML saved to: html_cilr/cilr_vs_hildebert_full.html
Table shape: (10416, 8)
Sentences: 791


In [6]:
# --- Extract the 100 most negative sentence rows and save a focused HTML ---
from IPython.display import IFrame
import re
import base64

html = result["colourcoded_text"]

# Escape <EOL> so it displays as text instead of being interpreted as HTML
html = html.replace("<EOL>", "&#60;EOL&#62;")

sentence_rows = re.findall(
    r'<div class="sentence-row">.*?</div>\s*</div>',
    html,
    re.DOTALL,
)
print(f"Total sentence rows: {len(sentence_rows)}")

# With order_by='importance', the final rows are the most negative.
last100 = sentence_rows[-100:]
last100.reverse()

renumbered = []
for i, row in enumerate(last100, 1):
    row = re.sub(
        r'<span class="sentence-number-circle"[^>]*>\d+</span>',
        '<span class="sentence-number-circle" '
        'style="background-color: #2166ac; color: white;">'
        + str(i)
        + "</span>",
        row,
    )
    renumbered.append(row)

output_html = (
    '<!DOCTYPE html><html lang="en"><head><meta charset="UTF-8">'
    '<link href="https://fonts.googleapis.com/css2?'
    'family=Roboto+Condensed:wght@300;400;700&display=swap" rel="stylesheet">'
    "<style>"
    "body{font-family:'Roboto Condensed',sans-serif;margin:0;padding:20px;"
    "background:#f5f5f5;font-weight:300}"
    "h2{text-align:center;color:#333;font-weight:700}"
    ".subtitle{text-align:center;color:#666;margin-bottom:20px;font-size:14px}"
    ".sentence-row{display:flex;align-items:flex-start;margin-bottom:12px;gap:10px}"
    ".sentence-column{flex:1;line-height:1.6;font-size:15px}"
    ".sentence-number-circle{display:inline-block;width:28px;height:28px;"
    "line-height:28px;border-radius:50%;text-align:center;font-size:12px;"
    "font-weight:700}"
    ".heading-row{display:flex;align-items:center;margin-bottom:10px;gap:10px}"
    ".heading-spacer{width:38px;flex-shrink:0}"
    ".column-heading{flex:1;text-align:center;color:#555;font-size:1.2em;"
    "font-weight:400;border-bottom:2px solid #ddd;padding-bottom:10px}"
    "</style></head><body>"
    "<h2>100 Most Dissimilar Sentences — Carmen in libros regum vs. Hildebert</h2>"
    "<p class='subtitle'>Token-level LambdaG scores with relative scale "
    "(most negative = least compatible with Hildebert's profile)</p>"
    '<div class="heading-row">'
    '<div class="heading-spacer"></div>'
    '<div class="column-heading">Masked (POS Tags)</div>'
    '<div class="column-heading">Unmasked (Latin Text)</div>'
    "</div>"
    + "\n".join(renumbered)
    + "</body></html>"
)

FOCUSED_HTML_PATH = OUTPUT_DIR / "cilr_vs_hildebert_100_most_dissimilar.html"
FOCUSED_HTML_PATH.write_text(output_html, encoding="utf-8")
print(f"Focused HTML saved to: {FOCUSED_HTML_PATH}")

# Display in an IFrame to avoid CSS conflicts with the notebook
html_bytes = output_html.encode("utf-8")
html_b64 = base64.b64encode(html_bytes).decode("utf-8")
IFrame(f"data:text/html;base64,{html_b64}", width="100%", height=800)


Total sentence rows: 791
Focused HTML saved to: html_cilr/cilr_vs_hildebert_100_most_dissimilar.html
